# DA6401 — Continue Training from Friend's Checkpoints (+5 epochs)

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Save & Run All (Commit)**

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available()

In [ ]:
import gdown

# Friend's Drive file IDs
CLASSIFIER_ID = "1Ob4ecaMOaoH-56TdIQ9lYEdGKfn1jy8L"
LOCALIZER_ID  = "1jRM83Xr6HETheUhv1MQqRMEl2eAwbkSK"
UNET_ID       = "1u-ux_IFhPResLG7TPl8u0qcJkcAvKLZL"

for drive_id, out_path, label in [
    (CLASSIFIER_ID, f"{CKPT}/classifier.pth", "classifier"),
    (LOCALIZER_ID,  f"{CKPT}/localizer.pth",  "localizer"),
    (UNET_ID,       f"{CKPT}/unet.pth",        "unet"),
]:
    print(f"Downloading {label}...")
    gdown.download(id=drive_id, output=out_path, quiet=False, fuzzy=True)
    mb = os.path.getsize(out_path) / 1e6 if os.path.exists(out_path) else 0
    print(f"  {'✓' if mb > 1 else '✗'} {mb:.0f} MB\n")

In [ ]:
import torch

# Inspect all three checkpoints — print all layer names and shapes
for name, path in [("classifier", f"{CKPT}/classifier.pth"),
                   ("localizer",  f"{CKPT}/localizer.pth"),
                   ("unet",       f"{CKPT}/unet.pth")]:
    sd = torch.load(path, map_location="cpu", weights_only=False)
    keys = list(sd.keys())
    total_params = sum(v.numel() for v in sd.values() if isinstance(v, torch.Tensor))
    print(f"\n{'='*55}")
    print(f"  {name}.pth — {total_params/1e6:.1f}M params, {len(keys)} tensors")
    print(f"{'='*55}")
    # Print first 5 and last 5 keys to understand structure
    for k in keys[:5]:
        print(f"  {k:50s} {list(sd[k].shape)}")
    if len(keys) > 10:
        print(f"  ... ({len(keys)-10} more) ...")
    for k in keys[-5:]:
        print(f"  {k:50s} {list(sd[k].shape)}")

In [ ]:
from data.pets_dataset import download_oxford_pet, create_dataloaders
download_oxford_pet("./data/oxford_pet")
train_loader, val_loader, _ = create_dataloaders(
    root="./data/oxford_pet", batch_size=32, num_workers=2
)
print("Dataset ready.")

## 1. Baseline scores on friend's weights

In [ ]:
from sklearn.metrics import f1_score
from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet
from train import compute_iou, compute_dice_score

device = torch.device("cuda")
results = {}

print("="*55)
print("  FRIEND'S WEIGHTS — BASELINE")
print("="*55)

# Classification
try:
    m = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
    m.load_state_dict(torch.load(f"{CKPT}/classifier.pth", map_location=device, weights_only=False))
    m.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in val_loader:
            preds.extend(m(b["image"].to(device)).argmax(1).cpu().tolist())
            labels.extend(b["label"].tolist())
    f1 = f1_score(labels, preds, average="macro")
    results["cls_f1"] = f1
    print(f"  Classification  F1  = {f1:.4f}  (need ≥ 0.80)  {'✓' if f1 >= 0.80 else '✗'}")
except Exception as e:
    print(f"  Classification  ERROR: {e}")

# Localization
try:
    m = VGG11Localizer().to(device)
    m.load_state_dict(torch.load(f"{CKPT}/localizer.pth", map_location=device, weights_only=False))
    m.eval()
    iou_sum, iou50, total = 0.0, 0, 0
    with torch.no_grad():
        for b in val_loader:
            ious = compute_iou(m(b["image"].to(device)).cpu(), b["bbox"])
            iou_sum += ious.sum().item(); iou50 += (ious>=0.5).sum().item(); total += len(ious)
    results["loc_iou"] = iou_sum/total; results["loc_acc"] = iou50/total
    print(f"  Localization    IoU = {iou_sum/total:.4f},  Acc@0.5 = {iou50/total*100:.1f}%  (need ≥ 60%)  {'✓' if iou50/total >= 0.6 else '✗'}")
except Exception as e:
    print(f"  Localization    ERROR: {e}")

# Segmentation
try:
    m = VGG11UNet(num_classes=3).to(device)
    m.load_state_dict(torch.load(f"{CKPT}/unet.pth", map_location=device, weights_only=False))
    m.eval()
    dice_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            pred = m(b["image"].to(device)).argmax(1).cpu()
            dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0)
            total += b["image"].size(0)
    results["seg_dice"] = dice_sum/total
    print(f"  Segmentation    Dice = {dice_sum/total:.4f}  (need ≥ 0.30)  {'✓' if dice_sum/total >= 0.30 else '✗'}")
except Exception as e:
    print(f"  Segmentation    ERROR: {e}")

print("="*55)

## 2. Continue Localizer (+5 epochs, lr=1e-4)

In [ ]:
import time
import torch.nn as nn
import wandb
from losses.iou_loss import IoULoss

EXTRA_EPOCHS = 5
LR = 1e-4

model = VGG11Localizer().to(device)
model.load_state_dict(torch.load(f"{CKPT}/localizer.pth", map_location=device, weights_only=False))

model.eval()
iou_sum, total = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        ious = compute_iou(model(b["image"].to(device)).cpu(), b["bbox"])
        iou_sum += ious.sum().item(); total += len(ious)
baseline_iou = iou_sum / total
print(f"Localizer baseline: {baseline_iou:.4f}")

wandb.init(project="da6401-assignment2", name="localizer_friends_5ep",
           tags=["continue", "localization"])

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EXTRA_EPOCHS, eta_min=1e-6)
mse_loss  = nn.MSELoss()
iou_loss  = IoULoss(reduction="mean")

best_iou = baseline_iou
for epoch in range(1, EXTRA_EPOCHS + 1):
    model.train()
    t0 = time.time()
    for b in train_loader:
        imgs, bboxes = b["image"].to(device), b["bbox"].to(device)
        optimizer.zero_grad()
        (mse_loss(model(imgs), bboxes) + iou_loss(model(imgs), bboxes)).backward()
        optimizer.step()
    scheduler.step()
    model.eval()
    iou_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            ious = compute_iou(model(b["image"].to(device)).cpu(), b["bbox"])
            iou_sum += ious.sum().item(); total += len(ious)
    val_iou = iou_sum / total
    wandb.log({"val/iou": val_iou})
    print(f"  Epoch {epoch}/{EXTRA_EPOCHS} | Val IoU: {val_iou:.4f} | {time.time()-t0:.0f}s")
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), f"{CKPT}/localizer.pth")
        print(f"    ✓ Saved!")

wandb.finish()
print(f"\nLocalizer: {baseline_iou:.4f} → {best_iou:.4f}")

## 3. Continue U-Net (+5 epochs, lr=5e-4, partial freeze)

In [ ]:
EXTRA_EPOCHS = 5
LR = 5e-4

seg_model = VGG11UNet(num_classes=3).to(device)
seg_model.load_state_dict(torch.load(f"{CKPT}/unet.pth", map_location=device, weights_only=False))

for name, param in seg_model.encoder.named_parameters():
    param.requires_grad = not any(name.startswith(b) for b in
                                  ["block0","block1","block2","pool0","pool1","pool2"])

seg_model.eval()
dice_sum, total = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        pred = seg_model(b["image"].to(device)).argmax(1).cpu()
        dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0); total += b["image"].size(0)
baseline_dice = dice_sum / total
print(f"U-Net baseline: {baseline_dice:.4f}")

wandb.init(project="da6401-assignment2", name="unet_friends_5ep",
           tags=["continue", "segmentation"])

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0,1.0,2.0]).to(device))
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, seg_model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EXTRA_EPOCHS, eta_min=1e-6)

best_dice = baseline_dice
for epoch in range(1, EXTRA_EPOCHS + 1):
    seg_model.train()
    t0 = time.time()
    for b in train_loader:
        imgs, masks = b["image"].to(device), b["mask"].to(device)
        optimizer.zero_grad()
        criterion(seg_model(imgs), masks).backward()
        optimizer.step()
    scheduler.step()
    seg_model.eval()
    dice_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            pred = seg_model(b["image"].to(device)).argmax(1).cpu()
            dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0); total += b["image"].size(0)
    val_dice = dice_sum / total
    wandb.log({"val/dice": val_dice})
    print(f"  Epoch {epoch}/{EXTRA_EPOCHS} | Val Dice: {val_dice:.4f} | {time.time()-t0:.0f}s")
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(seg_model.state_dict(), f"{CKPT}/unet.pth")
        print(f"    ✓ Saved!")

wandb.finish()
print(f"\nU-Net: {baseline_dice:.4f} → {best_dice:.4f}")

## 4. Summary

In [ ]:
print("="*55)
print("  OUTPUT FILES — download from Output tab")
print("="*55)
for f in sorted(os.listdir(CKPT)):
    mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
    print(f"  {f:30s}  {mb:.0f} MB")
print()
print("  NEXT: upload localizer.pth + unet.pth to Drive")
print("        paste new file IDs here to update multitask.py")
print("="*55)